## MCSBT_MIDTERM_ASSIGNMENT :: ROI IDEA to Word2Vec Modeling with Spark MLLib
**Yohannes Belay Woldeyes**

### ROI idea
Sales engineers lose a lot of time searching old documents to answer repetitive RFPs. That time could go to expert review and to bidding on more work. This notebook builds the search part of the idea: type in a requirement and get back the 50 closest matching notices, ranked by Word2Vec similarity.

### Dataset
EuroTender, public procurement notices from the EU's Tenders Electronic Daily, published on Hugging Face as `sinceaihq/eurotender-cpv` under CC-BY 4.0. I use a subset of 15,952 English competition notices from October 2025 to September 2026, saved as `rfp_ted_notices.csv`.

It has the same shape as the Crunchbase file: `notice_id` takes the place of `crunchbase_uuid`, `title` of `name`, and `description` of `short_description`.

These notices are written by buyers, not by vendors. Real proposals are private, so I use the notices as a stand-in for a company's own proposal archive. The method would be the same on the real archive.

### Running this notebook in Colab
Upload `rfp_ted_notices.csv` into the `sample_data` folder using the Files panel on the left, then run all cells.

### Install and initialize PySpark

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [3]:
spark = SparkSession.builder.getOrCreate()

In [4]:
print("Using Apache Spark Version", spark.version)

Using Apache Spark Version 4.0.4


### Process EuroTender RFP notices in Spark DataFrame

#### Read the csv file into Spark DataFrame
The text in this file has quotes written as `""`, so it also needs `escape` to be read correctly.

In [5]:
rfp = spark.read.options(inferSchema='True',delimiter=',',header='True',escape='"') \
           .csv("/content/sample_data/rfp_ted_notices.csv")
rfp.count()

15952

In [6]:
rfp.printSchema()

root
 |-- notice_id: string (nullable = true)
 |-- publication_date: date (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- contract_nature: string (nullable = true)
 |-- procedure_type: string (nullable = true)



#### Filter out empty descriptions from the dataframe

In [7]:
rfp = rfp.select('notice_id','title','description') \
         .filter('description is not null and description != ""')
rfp.count()

15952

#### Remove repeated titles
The result of this notebook is a list of titles, so each title should appear once. Assignment 1 counted the unique rows and Assignment 2 kept the unique titles; this is the same step in Spark.

In [8]:
rfp = rfp.dropDuplicates(['title'])
rfp.count()

15540

#### Create a new column, "inputText", for downstream NLP processes

In [9]:
df = rfp.withColumn('inputText', F.col('description'))

In [10]:
df.show(5)

+-----------+--------------------+--------------------+--------------------+
|  notice_id|               title|         description|           inputText|
+-----------+--------------------+--------------------+--------------------+
|425194-2026|"Learn to Ride" C...|"Learn to Ride" C...|"Learn to Ride" C...|
|662827-2025|"Safe City" video...|More detailed des...|More detailed des...|
| 31425-2026|(2025-132) Projec...|The competition i...|The competition i...|
| 84387-2026|(2025-134) Archit...|The competition i...|The competition i...|
|730944-2025|(2025-4244) KI Ro...|The competition i...|The competition i...|
+-----------+--------------------+--------------------+--------------------+
only showing top 5 rows


### Train a Word2Vec Model on the RFP notices

#### Apply Regex Tokenizer and StopWord Remover

In [11]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, Word2Vec

In [12]:
regextok = RegexTokenizer(gaps = False, pattern = r'\w+',
                          inputCol = 'inputText', outputCol = 'tokens')

In [13]:
stopwrmv = StopWordsRemover(inputCol = 'tokens', outputCol = 'tokens_sw_removed')

In [14]:
df = regextok.transform(df)
df = stopwrmv.transform(df)
df.select('notice_id','tokens','tokens_sw_removed').show(3, truncate = 70)

+-----------+----------------------------------------------------------------------+----------------------------------------------------------------------+
|  notice_id|                                                                tokens|                                                     tokens_sw_removed|
+-----------+----------------------------------------------------------------------+----------------------------------------------------------------------+
|425194-2026|[learn, to, ride, cycle, area, design, and, build, for, site, at, c...|[learn, ride, cycle, area, design, build, site, carrigduff, bunclod...|
|662827-2025|[more, detailed, description, of, the, subject, for, the, agreement...|[detailed, description, subject, agreement, public, procurement, gi...|
| 31425-2026|[the, competition, is, for, the, delivery, of, a, framework, agreem...|[competition, delivery, framework, agreement, project, administrati...|
+-----------+---------------------------------------------------

Total number of tokens before and after removing stopwords:

In [15]:
df.select(F.sum(F.size('tokens')).alias('tokens'),
          F.sum(F.size('tokens_sw_removed')).alias('tokens_sw_removed')).show()

+-------+-----------------+
| tokens|tokens_sw_removed|
+-------+-----------------+
|1309699|           833258|
+-------+-----------------+



#### Train Word2Vec Model on the RFP Dataset
Spark's Word2Vec uses Skip-Gram: each word is used to predict the words around it, and the weights learned while doing this become the word vectors.

The template uses the default of one training pass (`maxIter = 1`). On this dataset, that gave only 4 relevant titles in the top 50 for the cybersecurity query, which is a sign of underfitting, so I train for 10 passes. The seed is fixed so that running the notebook again on the same setup gives the same results.

In [16]:
word2vec = Word2Vec(vectorSize = 100, minCount = 5, maxIter = 10, seed = 42, inputCol = 'tokens_sw_removed', outputCol = 'wordvectors')

In [17]:
model = word2vec.fit(df)

#### Search for synonyms in the word2vec model based on similarity

In [18]:
synonyms = model.findSynonyms("software", 20)
synonyms.show()

+--------------+------------------+
|          word|        similarity|
+--------------+------------------+
|      hardware|0.7983289361000061|
|      licences|0.7659044861793518|
|      licenses|0.7107361555099487|
|      firmware|0.6593204736709595|
|      solution|0.6235293745994568|
|           hpe|0.6196330189704895|
|         cupps|0.6123427152633667|
|         cloud|0.6080964803695679|
|        citrix| 0.607541024684906|
|           pki|0.6051012277603149|
|        oracle|0.5950330495834351|
|         adobe|0.5872564911842346|
|           lsp|0.5759199261665344|
|       servers|0.5743181109428406|
|virtualisation|0.5645096302032471|
|          saas|0.5625397562980652|
|        gitlab|0.5527631044387817|
|   omnichannel|0.5516701340675354|
| configuration|0.5497519969940186|
|    components|0.5479697585105896|
+--------------+------------------+



In [19]:
synonyms = model.findSynonyms("security", 20)
synonyms.show()

+-------------+-------------------+
|         word|         similarity|
+-------------+-------------------+
|cybersecurity| 0.6181386709213257|
|          soc| 0.6079550385475159|
|    incidents| 0.6052743196487427|
|      privacy| 0.5931827425956726|
|       threat| 0.5927765369415283|
|     incident| 0.5888481736183167|
|        guard| 0.5885908007621765|
| preparedness| 0.5725288987159729|
|   escalation|  0.562544047832489|
|     guarding| 0.5561073422431946|
|      posture| 0.5504564642906189|
|         siem| 0.5469570755958557|
|        cefta| 0.5418372750282288|
|        cyber| 0.5359336733818054|
|   continuity| 0.5303893089294434|
|        crowd| 0.5177558660507202|
|          mdr| 0.5140237212181091|
|       bilbao| 0.5101361274719238|
|      threats|0.49878478050231934|
|      attacks| 0.4960135519504547|
+-------------+-------------------+



### Query the text using word2vec model

#### Reload the RFP data and create an inputText column

In [20]:
df = rfp.withColumn('inputText', F.col('description'))

#### Apply the regex tokenizer, stopword remover, and word2vec model

In [21]:
df = regextok.transform(df)
df = stopwrmv.transform(df)
df = model.transform(df)

In [22]:
df.select('notice_id','title','wordvectors').show(5)

+-----------+--------------------+--------------------+
|  notice_id|               title|         wordvectors|
+-----------+--------------------+--------------------+
|425194-2026|"Learn to Ride" C...|[0.04425648755927...|
|662827-2025|"Safe City" video...|[0.02402979321777...|
| 31425-2026|(2025-132) Projec...|[0.06672460353818...|
| 84387-2026|(2025-134) Archit...|[0.05394780667333...|
|730944-2025|(2025-4244) KI Ro...|[-0.1339153301003...|
+-----------+--------------------+--------------------+
only showing top 5 rows


In [23]:
df.count()

15540

#### Selecting all entries for querying
The dataset is small enough to use every notice.

In [24]:
chunks = df.select('notice_id','title','wordvectors').collect()
len(chunks)

15540

In [25]:
chunks[0]

Row(notice_id='425194-2026', title='"Learn to Ride" Cycle Area Design and Build for site at Carrigduff, Bunclody, Co. Wexford', wordvectors=DenseVector([0.0443, 0.1291, 0.0137, -0.0827, -0.1333, -0.0066, 0.1883, -0.061, 0.0564, 0.051, 0.1065, -0.0255, -0.0254, 0.192, -0.1734, -0.1307, 0.1743, 0.0043, -0.0458, 0.1066, 0.077, 0.188, 0.0887, 0.0792, -0.0013, -0.1185, -0.0253, 0.0584, -0.0304, 0.0096, -0.0585, -0.1199, -0.0371, -0.0343, -0.093, -0.0825, -0.0206, -0.0529, -0.0902, -0.0686, 0.1354, -0.0482, -0.0097, 0.0446, 0.1062, 0.1154, -0.0142, 0.2349, -0.0368, 0.0416, 0.0048, 0.0249, -0.0797, 0.1985, -0.0037, 0.1907, 0.0505, -0.0361, -0.1872, 0.0625, 0.1143, 0.05, 0.0909, 0.0816, 0.056, 0.0732, 0.0587, 0.0752, 0.0244, 0.034, 0.0339, -0.1363, -0.1469, -0.1092, 0.0986, -0.0278, -0.0353, 0.0744, -0.1173, 0.1171, 0.0276, 0.0272, 0.0146, 0.2552, -0.2033, -0.1332, 0.0218, -0.0733, 0.1969, -0.0895, -0.244, 0.0354, 0.0007, 0.0809, -0.0123, 0.105, -0.0226, 0.0855, 0.0724, -0.003]))

#### Create a search query in Spark

In [26]:
SEARCH_QUERY = "cybersecurity threat monitoring vulnerability assessment"
SEARCH_QUERY

'cybersecurity threat monitoring vulnerability assessment'

#### Create a Spark Dataframe for the query

In [27]:
query_df = spark.createDataFrame([(1, SEARCH_QUERY)]).toDF('index','inputText')

In [28]:
query_df.show(truncate = False)

+-----+--------------------------------------------------------+
|index|inputText                                               |
+-----+--------------------------------------------------------+
|1    |cybersecurity threat monitoring vulnerability assessment|
+-----+--------------------------------------------------------+



#### Transform the search query

In [29]:
query_tok = regextok.transform(query_df)
query_swr = stopwrmv.transform(query_tok)
query_vec = model.transform(query_swr)
query_vec.show()

+-----+--------------------+--------------------+--------------------+--------------------+
|index|           inputText|              tokens|   tokens_sw_removed|         wordvectors|
+-----+--------------------+--------------------+--------------------+--------------------+
|    1|cybersecurity thr...|[cybersecurity, t...|[cybersecurity, t...|[-0.1124761149287...|
+-----+--------------------+--------------------+--------------------+--------------------+



#### Extract the word vector for the search query

In [30]:
query_vec = query_vec.select('wordvectors').collect()[0][0]
query_vec

DenseVector([-0.1125, -0.1769, -0.1482, -0.095, 0.0204, -0.1554, 0.1531, -0.1712, -0.5837, -0.2382, 0.1547, 0.1573, 0.4369, 0.2322, -0.1768, -0.1147, -0.1073, 0.1232, 0.2508, 0.3416, 0.4364, -0.0402, -0.0044, -0.1811, 0.0254, 0.2567, 0.0331, -0.2, -0.0502, 0.0962, 0.0898, 0.307, -0.0034, -0.1294, 0.0785, -0.3562, 0.0436, -0.0512, 0.1478, 0.4142, -0.1045, -0.0143, -0.2896, -0.0189, 0.0293, 0.2081, -0.1672, 0.1934, 0.3708, 0.3657, -0.1648, -0.0409, -0.1294, 0.0705, -0.092, 0.249, -0.2944, 0.1158, -0.2765, -0.2215, -0.0951, -0.2215, 0.286, 0.003, -0.0019, -0.0457, -0.3893, -0.2017, -0.0804, 0.2899, 0.1874, -0.2546, 0.1114, -0.0018, -0.0215, 0.0115, -0.1661, -0.3345, -0.2503, 0.3407, -0.0752, 0.1798, -0.0099, -0.1856, 0.2565, -0.1417, -0.1541, -0.1486, 0.2365, 0.1415, -0.1361, 0.5313, -0.2823, -0.1138, -0.0102, -0.0481, -0.0807, 0.1948, 0.1141, 0.0684])

#### Calculate Cosine Similarity

In [31]:
import numpy as np
def cossim(v1, v2):
    if np.dot(v1, v1) == 0 or np.dot(v2, v2) == 0:
        return 0.0
    return float(np.dot(v1, v2) / np.sqrt(np.dot(v1, v1)) / (np.sqrt(np.dot(v2, v2))))

In [32]:
data = [(i[0], i[1], float(cossim(query_vec, i[2]))) for i in chunks]

In [33]:
sim_df = spark.createDataFrame(data).toDF('notice_id', 'title', 'similarity').orderBy(['similarity', 'notice_id'], ascending=[False, True])

#### Top 50 matching titles

In [34]:
sim_df.show(50, truncate = False)

+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|notice_id  |title                                                                                                                                                      |similarity        |
+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|514037-2026|Services for the Design, Development and Analysis of Cybersecurity Indexes                                                                                 |0.8223908730482643|
|523669-2026|Cyber Security Consultancy and Technical Services                                                                                                          |0.8101229828989122|
|15980-2026 |Provision of Security Operations Centre an

### Test and evaluate search queries

#### Search function
The same query steps as above in one function, as in the modular version of Assignment 2.

In [35]:
def search_titles(search_query):
    query_df = spark.createDataFrame([(1, search_query)]).toDF('index','inputText')
    query_tok = regextok.transform(query_df)
    query_swr = stopwrmv.transform(query_tok)
    query_vec = model.transform(query_swr).select('wordvectors').collect()[0][0]
    data = [(i[0], i[1], float(cossim(query_vec, i[2]))) for i in chunks]
    return spark.createDataFrame(data).toDF('notice_id', 'title', 'similarity').orderBy(['similarity', 'notice_id'], ascending=[False, True])

#### Second test query: top 50 matching titles

In [36]:
SEARCH_QUERY_2 = "cloud migration database infrastructure management"
sim_df_2 = search_titles(SEARCH_QUERY_2)
sim_df_2.show(50, truncate = False)

+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|notice_id  |title                                                                                                                                                                                                   |similarity        |
+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|511620-2026|Request for tender for the provision of Oracle Cloud@Customer Implementation, Managed Services and Anciliary Services                                                                                   |0.8443903472353772|
|858342-2025|DESC: Hosting and Support Services                 

#### Third test query: top 50 matching titles

In [37]:
SEARCH_QUERY_3 = "data protection privacy compliance"
sim_df_3 = search_titles(SEARCH_QUERY_3)
sim_df_3.show(50, truncate = False)

+-----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|notice_id  |title                                                                                                                                                                                                                   |similarity        |
+-----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|701535-2025|Outsourced Data Protection Service                                                                                                                                                                                      |0.8234684779356489|


#### Evaluate the three queries
The data has no labels saying which notice answers which query, so I use a simple check. Each query gets a list of topic terms, and a notice counts as relevant if its title or description contains one of them.

- **Precision at 50** is the share of the top 50 titles that are relevant.
- **Base rate** is the share of all notices that are relevant, which is what picking 50 notices at random would give on average.
- **Lift** is precision at 50 divided by the base rate. Above 1 means the search does better than chance.

This only checks for matching words and I chose the terms myself, so it is a rough measure and not a human judgement of each result.

In [38]:
TOPIC_TERMS_1 = r"cyber|information security|penetration test|vulnerabilit|\bsiem\b|\bsoc\b|firewall|incident response|intrusion|security operations"
TOPIC_TERMS_2 = r"\bcloud\b|data cent(er|re)|database|hosting|\bsaas\b|\biaas\b|\bpaas\b|\bservers?\b|data migration|\bsql\b"
TOPIC_TERMS_3 = r"data protection|gdpr|privacy|\bdpo\b|personal data"

In [39]:
def evaluate(search_query, sim_df, topic_terms):
    relevant = rfp.filter(F.lower(F.concat_ws(' ', 'title', 'description')).rlike(topic_terms)).select('notice_id')
    hits = sim_df.limit(50).join(relevant, 'notice_id').count()
    precision = hits / 50
    base_rate = relevant.count() / rfp.count()
    return (search_query, hits, precision, round(base_rate, 3), round(precision / base_rate, 1))

In [40]:
results = [evaluate(SEARCH_QUERY, sim_df, TOPIC_TERMS_1),
           evaluate(SEARCH_QUERY_2, sim_df_2, TOPIC_TERMS_2),
           evaluate(SEARCH_QUERY_3, sim_df_3, TOPIC_TERMS_3)]
spark.createDataFrame(results).toDF('query', 'relevant_in_top_50', 'precision_at_50', 'base_rate', 'lift').show(truncate = False)

+--------------------------------------------------------+------------------+---------------+---------+----+
|query                                                   |relevant_in_top_50|precision_at_50|base_rate|lift|
+--------------------------------------------------------+------------------+---------------+---------+----+
|cybersecurity threat monitoring vulnerability assessment|31                |0.62           |0.014    |45.2|
|cloud migration database infrastructure management      |39                |0.78           |0.033    |23.8|
|data protection privacy compliance                      |13                |0.26           |0.006    |43.9|
+--------------------------------------------------------+------------------+---------------+---------+----+



### Findings

#### Effect of training passes
Relevant titles in the top 50, using the keyword check above:

| Query | 1 pass | 10 passes |
|---|---|---|
| cybersecurity threat monitoring vulnerability assessment | 4 | 31 |
| cloud migration database infrastructure management | 32 | 39 |
| data protection privacy compliance | 10 | 13 |

One training pass underfit the data. With 10 passes, the top 8 results for the cybersecurity query are all about cybersecurity, and the words closest to "security" include "cybersecurity", "soc", "incidents" and "threat".

#### Reading the results
The keyword check is only a rough measure, so I also read the titles.

| Query | Keyword check | Reading the titles |
|---|---|---|
| Cybersecurity | 31 | About 31 |
| Cloud and database | 39 | About 21 |
| Data protection | 13 | 5 to 9 |

The cloud query's keyword count is too high because it also counts general IT systems, such as CRM, HR and print services. The data protection query is the weakest: only 0.6% of notices mention the topic, so most results are nearby security and compliance notices.

#### Limits
- Averaging word vectors loses the meaning of a word in context. "Security" also brings in guarding and travel risk notices, and the data protection query returned "Vulnerability Assessments to International Protection Applicants".
- A high similarity score does not guarantee a relevant notice, so the scores cannot be read as a measure of quality.
- These notices are requests from buyers, so the search finds similar requirements, not past answers.

#### What this means for the ROI idea
The search works as a shortlist for an engineer to review, not as a final answer. To reuse past answers, the same method would have to run on a company's own proposals.